In [1]:
#r "nuget:Microsoft.ML"
#r "nuget:XPlot.Plotly"

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed package Microsoft.ML version 1.7.1

Installed package XPlot.Plotly version 4.0.6

In [3]:
using Microsoft.ML;
using Microsoft.ML.Data;
using XPlot.Plotly;

In [4]:
public class TaxiTrip
{
[LoadColumn(0)]
public string VendorId;

[LoadColumn(1)]
public string RateCode;

[LoadColumn(2)]
public float PassengerCount;

[LoadColumn(3)]
public float TripTime;

[LoadColumn(4)]
public float TripDistance;

[LoadColumn(5)]
public string PaymentType;

[LoadColumn(6)]
public float FareAmount;
}

public class TaxiTripFarePrediction
{
[ColumnName("Score")]
public float Score;
}


In [32]:
display(h1("Code for loading the data into IDataViews: training dataset and test dataset"));

MLContext mlContext = new MLContext(seed: 0);
string TrainDataPath = "./taxi-fare-train.csv";
string TestDataPath = "./taxi-fare-test.csv";

IDataView trainDataView = mlContext.Data.LoadFromTextFile<TaxiTrip>(TrainDataPath, hasHeader: true, separatorChar: ';');
IDataView testDataView = mlContext.Data.LoadFromTextFile<TaxiTrip>(TestDataPath, hasHeader: true, separatorChar: ';');

display(h4("Schema of training DataView:"));
display(trainDataView.Schema);

Code for loading the data into IDataViews: training dataset and test dataset

Schema of training DataView:

index,Name,Index,IsHidden,Type,Annotations
0,VendorId,0,False,{ Microsoft.ML.Data.TextDataViewType: RawType: System.ReadOnlyMemory<System.Char> },{ Microsoft.ML.DataViewSchema+Annotations: Schema: [ ] }
1,RateCode,1,False,{ Microsoft.ML.Data.TextDataViewType: RawType: System.ReadOnlyMemory<System.Char> },{ Microsoft.ML.DataViewSchema+Annotations: Schema: [ ] }
2,PassengerCount,2,False,{ Microsoft.ML.Data.NumberDataViewType: RawType: System.Single },{ Microsoft.ML.DataViewSchema+Annotations: Schema: [ ] }
3,TripTime,3,False,{ Microsoft.ML.Data.NumberDataViewType: RawType: System.Single },{ Microsoft.ML.DataViewSchema+Annotations: Schema: [ ] }
4,TripDistance,4,False,{ Microsoft.ML.Data.NumberDataViewType: RawType: System.Single },{ Microsoft.ML.DataViewSchema+Annotations: Schema: [ ] }
5,PaymentType,5,False,{ Microsoft.ML.Data.TextDataViewType: RawType: System.ReadOnlyMemory<System.Char> },{ Microsoft.ML.DataViewSchema+Annotations: Schema: [ ] }
6,FareAmount,6,False,{ Microsoft.ML.Data.NumberDataViewType: RawType: System.Single },{ Microsoft.ML.DataViewSchema+Annotations: Schema: [ ] }


In [36]:
public static List<TaxiTrip> Head(MLContext mlContext, IDataView dataView, int numberOfRows = 4) 
{
    string msg = string.Format("DataView: Showing {0} rows with the columns", numberOfRows.ToString());
    display(msg);

    var rows = mlContext.Data.CreateEnumerable<TaxiTrip>(dataView, reuseRowObject: false)
            .Take(numberOfRows)
            .ToList();
    return rows;
}
display(h4("Showing a few rows from training DataView:"));
var fewRows = Head(mlContext, testDataView, 20);
display(fewRows);

Showing a few rows from training DataView:

DataView: Showing 20 rows with the columns

index,VendorId,RateCode,PassengerCount,TripTime,TripDistance,PaymentType,FareAmount
0,VTS,1,1,1140,3.75,CRD,15.5
1,VTS,1,1,480,2.72,CRD,10
2,VTS,1,1,1680,7.8,CSH,26.5
3,VTS,1,1,600,4.73,CSH,14.5
4,VTS,1,1,600,2.18,CRD,9.5
5,VTS,1,1,1260,10.33,CSH,29.5
6,VTS,1,1,600,2.01,CSH,9
7,VTS,1,1,480,1.5,CRD,7.5
8,VTS,1,1,660,2.49,CSH,10.5
9,VTS,1,1,360,1.13,CRD,6


In [38]:
var faresHistogram = Chart.Plot(new Graph.Histogram(){x = fares, autobinx = false, nbinsx = 20});
var layout = new Layout.Layout(){title="Distribution of taxi trips per cost"};
faresHistogram.WithLayout(layout);
faresHistogram.WithXTitle("Fare ranges");
faresHistogram.WithYTitle("Number of trips");
display(faresHistogram);


In [40]:
var chartFareVsdist = Chart.Plot(
    new Graph.Scatter()
    {
        x = distances,
        y = fares,
        mode = "markers",
        marker = new Graph.Marker()
        {
            color = times,
            colorscale = "Jet"
        }
    }
);
var layout = new Layout.Layout(){title="Plot Fare depending on Distance"};
chartFareVsdist.WithLayout(layout);
chartFareVsdist.Width = 500;
chartFareVsdist.Height = 500;
chartFareVsdist.WithXTitle("Distance");
chartFareVsdist.WithYTitle("Fares");
chartFareVsdist.WithLegend(false);
display(chartFareVsdist);

In [41]:
display(h1("Apply Data Transformations pipeline"));

// STEP 2: Common data process configuration with pipeline data transformations
var dataProcessPipeline = mlContext.Transforms.Categorical.OneHotEncoding(outputColumnName: "VendorIdEncoded", 
                                                                          inputColumnName: nameof(TaxiTrip.VendorId))
     .Append(mlContext.Transforms.Categorical.OneHotEncoding(outputColumnName: "RateCodeEncoded", 
                                                             inputColumnName: nameof(TaxiTrip.RateCode)))
     .Append(mlContext.Transforms.Categorical.OneHotEncoding(outputColumnName: "PaymentTypeEncoded",
                                                             inputColumnName: nameof(TaxiTrip.PaymentType)))
     .Append(mlContext.Transforms.NormalizeMeanVariance(outputColumnName: nameof(TaxiTrip.PassengerCount)))
     .Append(mlContext.Transforms.NormalizeMeanVariance(outputColumnName: nameof(TaxiTrip.TripTime)))
     .Append(mlContext.Transforms.NormalizeMeanVariance(outputColumnName: nameof(TaxiTrip.TripDistance)))
     .Append(mlContext.Transforms.Concatenate("Features", "VendorIdEncoded", "RateCodeEncoded", "PaymentTypeEncoded",
                                              nameof(TaxiTrip.PassengerCount), nameof(TaxiTrip.TripTime), 
                                              nameof(TaxiTrip.TripDistance)));

Apply Data Transformations pipeline

In [46]:
%%time
display(h1("Buibld Training Pipeline and Train the model"));
display(h4("Creating the Training Pipeline with trainer/algorithm"));
var trainer = mlContext.Regression.Trainers.Sdca(labelColumnName: "FareAmount", featureColumnName: "Features");
var trainingPipeline = dataProcessPipeline.Append(trainer);

var trainedModel = trainingPipeline.Fit(trainDataView);

Unhandled exception: (1,1): error CS1525: Invalid expression term '%'
(1,2): error CS1525: Invalid expression term '%'
(1,7): error CS1002: ; expected